# ConvLSTM data prep: HUC10 → model-ready zarr

Uses each **HUC10 shape** (`Huc10_in_{id}.geojson`) from S3 to clip/sample SWE, precipitation, temperature, and elevation onto a 4 km grid, then pads to a fixed size, masks outside the watershed, and writes one **zarr** per HUC10 to S3.

**Prerequisites:** Run `get_huc10_shapes.ipynb` first so that `huc10_shapes/` and `huc10_shapes/metadata/huc10_list.csv` exist on S3.

## 1. Config — change these as needed

All paths and parameters used in the pipeline. Edit this cell and re-run before running the rest of the notebook.

In [1]:
# --- S3: shapes and output ---
BUCKET = "convlstm-model-ready"
S3_PREFIX_SHAPES = "huc10_shapes"
S3_PREFIX_METADATA = "huc10_shapes/metadata"
S3_PREFIX_HUC12_IN_HUC10 = "huc12_shapes_in_huc10"  # HUC12-in-HUC10 GeoJSON files
S3_PREFIX_DATA = "model_reday_data"  # output zarrs: s3://BUCKET/model_reday_data/huc10_{id}_{Hmax}x{Wmax}.zarr

# Shape file naming (must match get_huc10_shapes.ipynb)
SHAPE_FNAME = "Huc10_in_{huc10_id}.geojson"  # e.g. Huc10_in_1711000507.geojson

# --- Source data (zarr on S3) ---
BRONZE_BUCKET = "s3://snowml-bronze"
SWE_ZARR = f"{BRONZE_BUCKET}/swe_all.zarr"
PR_ZARR = f"{BRONZE_BUCKET}/pr_all.zarr"
TMMN_ZARR = f"{BRONZE_BUCKET}/tmmn_all.zarr"
TMMX_ZARR = f"{BRONZE_BUCKET}/tmmx_all.zarr"
SNOW_CLASS_ZARR = f"{BRONZE_BUCKET}/snow_class_data.zarr"

# --- Grid and padding ---
PIXEL_KM = 4  # grid resolution (km); pixel size = 4000 m
HMAX, WMAX = 24, 24  # padded output size (height x width)

# --- Which HUC10s to process ---
# Set to None to load from metadata CSV; or pass a list e.g. ["1711000507", "1702000901"]
HUC10_IDS_OVERRIDE = None   # or ["1711000507"] for a single test run

## 2. Imports

In [2]:
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from rasterio import features
from rasterio.transform import from_origin
from rasterio.enums import Resampling
from pyproj import Transformer
from pyproj import CRS as PyCRS
import s3fs
import easysnowdata as easy

# Use WKT to avoid rasterio CRS-from-EPSG database lookup issues
CRS_5070_WKT = PyCRS.from_epsg(5070).to_wkt()

fs = s3fs.S3FileSystem(anon=False)

/Users/khaja/Documents/SnowML/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load list of HUC10 IDs to process

Reads `huc10_list.csv` from S3 (written by `get_huc10_shapes.ipynb`). If `HUC10_IDS_OVERRIDE` is set in config, that list is used instead.

In [3]:
if HUC10_IDS_OVERRIDE is not None:
    huc10_ids = [str(x) for x in HUC10_IDS_OVERRIDE]
    print(f"Using override: {len(huc10_ids)} HUC10(s)")
else:
    csv_path = f"s3://{BUCKET}/{S3_PREFIX_METADATA}/huc10_list.csv"
    with fs.open(csv_path) as f:
        df = pd.read_csv(f)
    huc10_ids = df["HUC10_ID"].astype(str).tolist()
    print(f"Loaded {len(huc10_ids)} HUC10 IDs from {csv_path}")

print("First 3:", huc10_ids[:3])

Loaded 91 HUC10 IDs from s3://convlstm-model-ready/huc10_shapes/metadata/huc10_list.csv
First 3: ['1702000901', '1702000902', '1702000903']


## 4. Helpers: load shape, build mask/grid, pad, mask, sample

- **load_shape**: Read one GeoJSON from S3, set CRS 4326, reproject to EPSG:5070 (Albers); returns `gdf_wgs`, `gdf_5070`.
- **build_mask_and_grid**: Rasterize polygon to (H,W), compute pixel-center coords in 5070 and lat/lon; takes `gdf_5070`.
- **sample_to_grid**: Interpolate a data variable onto our (y,x) lat/lon grid using xarray’s `interp`.
- **pad_static**: Center a 2D (H,W) array in (Hmax,Wmax) for static variables; returns padded array, top, left.
- **pad_temporal**: Center a 3D (day,H,W) array in (day,Hmax,Wmax) for temporal variables; takes top, left.
- **apply_mask_static**: Set to NaN where mask is 0 for 2D arrays (elev, lat, lon, x5070, y5070).
- **apply_mask_temporal**: Set to NaN where mask is 0 for 3D (day,y,x) arrays.
- **rasterize_huc12_ids**: Load `Huc12_in_<HUC10>.geojson` from S3 and rasterize full HUC12 IDs onto the H×W grid.

In [4]:
def load_shape(fs, bucket, prefix, huc10_id, fname_tpl=None):
    """Load one HUC10 GeoJSON from S3; return GDF in EPSG:4326 and EPSG:5070."""
    if fname_tpl is None:
        fname_tpl = "Huc10_in_{huc10_id}.geojson"
    fname = fname_tpl.format(huc10_id=huc10_id)
    key = f"{prefix}/{fname}"
    path = f"s3://{bucket}/{key}"
    with fs.open(path) as f:
        gdf_wgs = gpd.read_file(f).set_crs("EPSG:4326")
    gdf_5070 = gdf_wgs.to_crs("EPSG:5070")
    return gdf_wgs, gdf_5070

def build_mask_and_grid(gdf_5070, pixel_m=4000):
    """
    Rasterize polygon to (H,W), build transform, pixel centers (Xc,Yc) and (Lon,Lat).
    Returns: mask (H,W), H, W, Xc, Yc, Lon, Lat, transform.
    """
    xmin, ymin, xmax, ymax = gdf_5070.total_bounds
    W = math.ceil((xmax - xmin) / pixel_m)
    H = math.ceil((ymax - ymin) / pixel_m)
    transform = from_origin(xmin, ymax, pixel_m, pixel_m)
    geom = gdf_5070.geometry.iloc[0]
    mask = features.rasterize(
        [(geom, 1)], out_shape=(H, W), transform=transform,
        fill=0, dtype=np.uint8, all_touched=False
    )
    # Pixel centers in EPSG:5070 (meters)
    xs = xmin + (np.arange(W) + 0.5) * pixel_m
    ys = ymax - (np.arange(H) + 0.5) * pixel_m
    Xc, Yc = np.meshgrid(xs, ys)
    to_lonlat = Transformer.from_crs("EPSG:5070", "EPSG:4326", always_xy=True)
    Lon, Lat = to_lonlat.transform(Xc, Yc)
    return mask, H, W, Xc, Yc, Lon, Lat, transform

def sample_to_grid(da, lat_q, lon_q):
    """Interpolate DataArray onto (y,x) grid at (lat_q, lon_q); dims (y,x) or (day,y,x)."""
    """ Returns DataArray with dims (day, y, x) sampled at (Lat,Lon) grid centers using bilinear interpolation."""
    return da.interp(lat=lat_q, lon=lon_q, method="linear")

def pad_static(arr_hw, Hmax, Wmax, fill_value=0):
    """Pad a 2D (H,W) array (static variable) centered in (Hmax,Wmax). Returns padded array, top, left."""
    H, W = arr_hw.shape
    out = np.full((Hmax, Wmax), fill_value, dtype=arr_hw.dtype)
    top = (Hmax - H) // 2
    left = (Wmax - W) // 2
    out[top:top+H, left:left+W] = arr_hw
    return out, top, left

def pad_temporal(arr_day_hw, Hmax, Wmax, top, left, fill_value=np.nan):
    """Pad a 3D (day,H,W) array (temporal variable) into (day,Hmax,Wmax) with data at [top:top+H, left:left+W]."""
    n_day, H, W = arr_day_hw.shape
    out = np.full((n_day, Hmax, Wmax), fill_value, dtype=arr_day_hw.dtype)
    out[:, top:top+H, left:left+W] = arr_day_hw
    return out

def apply_mask_static(mask, arr_hw):
    """For static 2D (y,x): set to NaN where mask is 0. mask is 0/1 (e.g. uint8)."""
    return np.where(mask == 1, arr_hw, np.nan)

def apply_mask_temporal(mask, arr_day_y_x):
    """For temporal 3D (day,y,x): set to NaN where mask is 0. mask is 0/1 (e.g. uint8)."""
    return np.where(mask[None, :, :] == 1, arr_day_y_x, np.nan)


def rasterize_huc12_ids(fs, bucket, prefix_huc12, huc10_id, transform, H, W):
    """Load precomputed HUC12-in-HUC10 GeoJSON from S3 and rasterize HUC12 IDs to the HxW grid.

    Returns an (H, W) int64 numpy array where each in-basin pixel has the full numeric HUC12 code
    (12-digit integer), and 0 indicates no HUC12 / outside coverage.
    """
    fname12 = f"Huc12_in_{huc10_id}.geojson"
    key12 = f"{prefix_huc12}/{fname12}"
    path12 = f"s3://{bucket}/{key12}"
    with fs.open(path12) as f:
        gdf12 = gpd.read_file(f).set_crs("EPSG:4326")
    gdf12_5070 = gdf12.to_crs("EPSG:5070")
    shapes = [
        (geom, int(str(hid)))
        for geom, hid in zip(gdf12_5070.geometry, gdf12_5070["huc_id"])
    ]
    huc12_grid = features.rasterize(
        shapes,
        out_shape=(H, W),
        transform=transform,
        fill=0,
        dtype="int64",
        all_touched=False,
    )
    return huc12_grid

## 5. Open source zarrs and derive date range

SWE, precipitation, and min/max temperature are read from the bronze zarrs. The **common time window** (`common_start`, `common_end`) is computed from the data as the intersection of the day ranges of all four datasets—no fixed dates in config. We use this range when sampling and use the same grid (lat_q, lon_q) per HUC10 when calling `sample_to_grid`.

In [5]:
# Open once; we'll re-use for each HUC10 (sample_to_grid uses per-HUC10 lat_q, lon_q)
swe_ds = xr.open_zarr(fs.get_mapper(SWE_ZARR), consolidated=True)
pr_ds = xr.open_zarr(fs.get_mapper(PR_ZARR), consolidated=True)
tmmn_ds = xr.open_zarr(fs.get_mapper(TMMN_ZARR), consolidated=True)
tmmx_ds = xr.open_zarr(fs.get_mapper(TMMX_ZARR), consolidated=True)
snow_ds = xr.open_zarr(fs.get_mapper(SNOW_CLASS_ZARR), consolidated=True)

# Derive common date range from data (intersection of all four datasets)
def day_bounds(ds, day_dim="day"):
    d = ds[day_dim].values
    return d.min(), d.max()
swe_start, swe_end = day_bounds(swe_ds)
pr_start, pr_end = day_bounds(pr_ds)
tn_start, tn_end = day_bounds(tmmn_ds)
tx_start, tx_end = day_bounds(tmmx_ds)
common_start = max(swe_start, pr_start, tn_start, tx_start)
common_end = min(swe_end, pr_end, tn_end, tx_end)
print("SWE:", swe_ds.dims)
print("PR:", pr_ds.dims)
print("Common day range (from data):", common_start, "→", common_end)

SWE: FrozenMappingWarningOnValuesAccess({'day': 14610, 'lat': 621, 'lon': 1405})
PR: FrozenMappingWarningOnValuesAccess({'day': 15423, 'lat': 585, 'lon': 1386, 'crs': 1})
Common day range (from data): 1983-01-01T00:00:00.000000000 → 2022-09-30T00:00:00.000000000


In [8]:
swe_ds

<xarray.Dataset> Size: 51GB
Dimensions:  (day: 14610, lat: 621, lon: 1405)
Coordinates:
  * day      (day) datetime64[ns] 117kB 1982-10-01 1982-10-02 ... 2022-09-30
  * lat      (lat) float32 2kB 24.08 24.12 24.17 24.21 ... 49.83 49.88 49.92
  * lon      (lon) float32 6kB -125.0 -125.0 -124.9 ... -66.58 -66.54 -66.5
Data variables:
    SWE      (day, lat, lon) float32 51GB dask.array<chunksize=(365, 104, 235), meta=np.ndarray>

In [13]:
arr = swe_ds["SWE"].isel(day=0).values
valid = arr[~np.isnan(arr)]
print(max(valid) if len(valid) > 0 else "all NaN")

1485.0


## 6. Process one HUC10: shape → grid → sample → pad → mask → zarr

For each HUC10 we: load shape, build mask and (Lat, Lon) grid, sample SWE/PR/TMMN/TMMX/SnowClass, compute swe_today/swe_lag30/swe_target and tavg, fetch Copernicus DEM via easysnowdata and aggregate to the 4 km grid, pad everything to (HMAX,WMAX), apply mask, then write an xarray Dataset to zarr. 

### What `process_one_huc10` does (simple overview)

**Goal:** For one watershed (HUC10), we build a small **grid of 4 km pixels** that covers it, fill that grid with SWE, precipitation, temperature, elevation, a static snow-class map, and a static HUC12 ID map, then **pad** the grid to a fixed 24×24 size (so every HUC10 output has the same shape for the model), **mask** out pixels outside the watershed, and **save** one zarr per HUC10.

**Steps in plain words:**
1. **Load the shape** — GeoJSON polygon of the HUC10 in lat/lon and in Albers (EPSG:5070).
2. **Build the grid** — Figure out how many pixels (H×W) cover the watershed at 4 km, rasterize the polygon to a 0/1 mask, and compute the **center** of each pixel in meters (x,y) and in lat/lon.
3. **Sample the data** — For each day, interpolate SWE, precipitation, and min/max temperature from the big zarrs onto our pixel centers; compute average temperature and SWE “today”, “30 days ago”, and “tomorrow” (target). In addition, sample the **static snow-classification grid** once to get an integer `snow_class` per pixel, and rasterize the precomputed HUC12 polygons to get an integer `huc12_id` per pixel.
4. **Trim days** — Keep only days where we have both a 30‑day lag and a next-day target (drop first 30 and last 1).
5. **Get elevation** — Download Copernicus DEM for the HUC’s bounding box, clip to the polygon, reproject to the same grid, and average into our 4 km pixels.
6. **Pad** — Put the H×W grids (mask, elevation, snow_class, huc12_id, lat, lon, x5070, y5070, and all time series) in the **center** of a 24×24 canvas; rest is fill (0 or NaN).
7. **Mask** — Set all values outside the watershed (where mask is 0) to NaN (or 0 for IDs) so the model ignores them.
8. **Save** — Write an xarray Dataset (with lat, lon, x5070, y5070, mask, elev, snow_class, huc12_id, pr, tavg, swe_lag30, swe_today, swe_target) to zarr on S3.

In [70]:
def process_one_huc10(huc10_id):
    """Load shape, build grid, sample SWE/PR/tavg, get elevation (DEM), pad, mask, write zarr. Output matches Cap0."""
    pixel_m = PIXEL_KM * 1000
    # Load shape
    gdf_wgs, gdf_5070 = load_shape(fs, BUCKET, S3_PREFIX_SHAPES, huc10_id, fname_tpl=SHAPE_FNAME)
    # Build mask and grid
    mask, H, W, Xc, Yc, Lon, Lat, transform = build_mask_and_grid(gdf_5070, pixel_m)
    xmin, ymax = transform.xoff, transform.yoff
    lat_q = xr.DataArray(Lat, dims=["y", "x"])
    lon_q = xr.DataArray(Lon, dims=["y", "x"])
    # Sample to grid (day, y, x)
    swe_hw = sample_to_grid(swe_ds["SWE"], lat_q, lon_q).sel(day=slice(common_start, common_end))
    pr_hw = sample_to_grid(pr_ds["precipitation_amount"], lat_q, lon_q).sel(day=slice(common_start, common_end))
    tmmn_hw = sample_to_grid(tmmn_ds["air_temperature"], lat_q, lon_q).sel(day=slice(common_start, common_end))
    tmmx_hw = sample_to_grid(tmmx_ds["air_temperature"], lat_q, lon_q).sel(day=slice(common_start, common_end))
    tavg_hw = (tmmn_hw + tmmx_hw) / 2.0
    swe_today = swe_hw
    swe_lag30 = swe_hw.shift(day=30)
    swe_target = swe_hw.shift(day=-1)
    # Align to valid days (drop first 30 and last 1)
    valid = np.arange(30, swe_hw.sizes["day"] - 1)
    swe_today = swe_today.isel(day=valid)
    swe_lag30 = swe_lag30.isel(day=valid)
    swe_target = swe_target.isel(day=valid)
    pr_hw = pr_hw.isel(day=valid)
    tavg_hw = tavg_hw.isel(day=valid)
    days_out = swe_today["day"].values
    # Elevation: Copernicus DEM → reproject to 5070 → reproject_match to 4km grid, then mask by HUC
    bbox_latlon = tuple(gdf_wgs.total_bounds)
    dem_latlong = easy.topography.get_copernicus_dem(bbox_input=bbox_latlon, resolution=90)
    dem_flat = dem_latlong.rio.reproject(CRS_5070_WKT)
    x_coords = xmin + (np.arange(W) + 0.5) * pixel_m
    y_coords = ymax - (np.arange(H) + 0.5) * pixel_m
    template = xr.DataArray(
        np.zeros((H, W), dtype=np.float32), dims=("y", "x"), coords={"y": y_coords, "x": x_coords}
    )
    template = template.rio.set_spatial_dims(x_dim="x", y_dim="y").rio.write_crs(CRS_5070_WKT).rio.write_transform(transform)
    elev_hw = dem_flat.rio.reproject_match(template, resampling=Resampling.average) # reproject dem to 4km grid called transform.

    # Static snow-classification: sample once per pixel
    snow_hw = sample_to_grid(snow_ds["SnowClass"], lat_q, lon_q)

    # Static HUC12 IDs: rasterize precomputed HUC12 polygons to the same grid (int64 IDs)
    huc12_hw = rasterize_huc12_ids(fs=fs, bucket=BUCKET,
        prefix_huc12=S3_PREFIX_HUC12_IN_HUC10,
        huc10_id=huc10_id, transform=transform, H=H, W=W,
    )

    # Padding and masking
    # Pad all static variables to (HMAX, WMAX)
    mask_pad, top0, left0 = pad_static(mask, HMAX, WMAX, fill_value=0)
    elev_pad, _, _ = pad_static(elev_hw.values.astype(np.float32), HMAX, WMAX, fill_value=np.nan)
    Lon_pad, _, _ = pad_static(Lon.astype(np.float32), HMAX, WMAX, fill_value=np.nan)
    Lat_pad, _, _ = pad_static(Lat.astype(np.float32), HMAX, WMAX, fill_value=np.nan)
    Xc_pad, _, _ = pad_static(Xc.astype(np.float32), HMAX, WMAX, fill_value=np.nan)
    Yc_pad, _, _ = pad_static(Yc.astype(np.float32), HMAX, WMAX, fill_value=np.nan)
    snow_pad, _, _ = pad_static(snow_hw.values.astype(np.int8), HMAX, WMAX, fill_value=0)
    huc12_pad, _, _ = pad_static(huc12_hw.astype("int64"), HMAX, WMAX, fill_value=0)

    # Pad temporal variables then apply mask (mask_pad is 0/1 uint8)
    pr_pad = apply_mask_temporal(mask_pad, pad_temporal(pr_hw.values.astype(np.float32), HMAX, WMAX, top0, left0))
    tavg_pad = apply_mask_temporal(mask_pad, pad_temporal(tavg_hw.values.astype(np.float32), HMAX, WMAX, top0, left0))
    swe_today_pad = apply_mask_temporal(mask_pad, pad_temporal(swe_today.values.astype(np.float32), HMAX, WMAX, top0, left0))
    swe_lag30_pad = apply_mask_temporal(mask_pad, pad_temporal(swe_lag30.values.astype(np.float32), HMAX, WMAX, top0, left0))
    swe_target_pad = apply_mask_temporal(mask_pad, pad_temporal(swe_target.values.astype(np.float32), HMAX, WMAX, top0, left0))

    # Apply mask to all static variables 
    elev_pad = apply_mask_static(mask_pad, elev_pad)
    Lat_pad = apply_mask_static(mask_pad, Lat_pad.astype(np.float32))
    Lon_pad = apply_mask_static(mask_pad, Lon_pad.astype(np.float32))
    Xc_pad = apply_mask_static(mask_pad, Xc_pad.astype(np.float32))
    Yc_pad = apply_mask_static(mask_pad, Yc_pad.astype(np.float32))
    snow_pad = np.where(mask_pad == 1, snow_pad, 0).astype(np.int8)
    huc12_pad = np.where(mask_pad == 1, huc12_pad, 0).astype("int64")

    # valid_swe: 1 where (inside HUC10 and no day is NaN), 0 if outside or any day is NaN
    has_valid_swe = np.all(~np.isnan(swe_today_pad), axis=0)
    valid_swe = ((mask_pad == 1) & has_valid_swe).astype(np.uint8)

    # Build dataset
    y_idx = np.arange(HMAX, dtype=np.int32)
    x_idx = np.arange(WMAX, dtype=np.int32)
    ds = xr.Dataset(
        data_vars=dict(
            lat=(("y", "x"), Lat_pad.astype(np.float32)),
            lon=(("y", "x"), Lon_pad.astype(np.float32)),
            x5070=(("y", "x"), Xc_pad.astype(np.float32)),
            y5070=(("y", "x"), Yc_pad.astype(np.float32)),
            mask=(("y", "x"), mask_pad.astype(np.uint8)),
            valid_swe=(("y", "x"), valid_swe),
            elev=(("y", "x"), elev_pad),
            snow_class=(("y", "x"), snow_pad),
            huc12_id=(("y", "x"), huc12_pad),
            pr=(("day", "y", "x"), pr_pad),
            tavg=(("day", "y", "x"), tavg_pad),
            swe_lag30=(("day", "y", "x"), swe_lag30_pad),
            swe_today=(("day", "y", "x"), swe_today_pad),
            swe_target=(("day", "y", "x"), swe_target_pad),
        ),
        coords=dict(day=days_out, y=y_idx, x=x_idx),
        attrs=dict(
            huc10_id=str(huc10_id),
            huc10_height=int(H), huc10_width=int(W),
            final_size_height=int(HMAX), final_size_width=int(WMAX),
            placement_top=int(top0), placement_left=int(left0),
            pixel_HnW_km=int(PIXEL_KM), crs="EPSG:5070",
            snow_class_flag_values=[1, 2, 3, 4, 5, 6, 7, 8],
            snow_class_flag_meanings="tundra boreal_forest maritime ephemeral prairie montane_forest ice ocean",
            huc12_id_note="int64 HUC12 code per pixel; 0 = outside HUC10 / no HUC12 id.",
            valid_swe_note="1 = inside HUC10 and no day is NaN at this pixel; 0 = outside or at least one day is NaN.",
            note="Values sampled by bilinear interpolation at 4km pixel centers; masked outside HUC10.",
        ),
    )
    out_key = f"{S3_PREFIX_DATA}/huc10_{huc10_id}_{HMAX}x{WMAX}.zarr"
    out_path = f"s3://{BUCKET}/{out_key}"
    ds.to_zarr(fs.get_mapper(out_path), mode="w")
    print(f"Wrote {out_path}")
    return out_path

## 7. Run for all HUC10s (or a subset)

Iterate over `huc10_ids` and call `process_one_huc10`. For a quick test, set `HUC10_IDS_OVERRIDE = ["1711000507"]` in the config cell and re-run.

In [71]:
def output_zarr_exists(huc10_id: str) -> bool:
    """Return True if this HUC10 output zarr already exists on S3."""
    out_key = f"{S3_PREFIX_DATA}/huc10_{huc10_id}_{HMAX}x{WMAX}.zarr"
    base = f"s3://{BUCKET}/{out_key}"
    # Zarr v3 marker
    if fs.exists(f"{base}/zarr.json"):
        return True
    # Zarr v2 markers
    if fs.exists(f"{base}/.zgroup") or fs.exists(f"{base}/.zmetadata"):
        return True
    return False

done = []
skipped = []
remaining = []
for huc10_id in huc10_ids:
    if output_zarr_exists(str(huc10_id)):
        skipped.append(str(huc10_id))
    else:
        remaining.append(str(huc10_id))
print(f"Already on S3: {len(skipped)}; remaining: {len(remaining)}")

for huc10_id in remaining:
    try:
        path = process_one_huc10(huc10_id)
        done.append((huc10_id, path))
    except Exception as e:
        print(f"Failed {huc10_id}: {e}")
print(f"Newly done: {len(done)}; total done now: {len(skipped)+len(done)} / {len(huc10_ids)}")

Already on S3: 0; remaining: 91


/Users/khaja/Documents/SnowML/.venv311/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Wrote s3://convlstm-model-ready/model_reday_data/huc10_1702000901_24x24.zarr


/Users/khaja/Documents/SnowML/.venv311/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Wrote s3://convlstm-model-ready/model_reday_data/huc10_1702000902_24x24.zarr


...

/Users/khaja/Documents/SnowML/.venv311/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Wrote s3://convlstm-model-ready/model_reday_data/huc10_1706020804_24x24.zarr


/Users/khaja/Documents/SnowML/.venv311/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Wrote s3://convlstm-model-ready/model_reday_data/huc10_1706020805_24x24.zarr
Wrote s3://convlstm-model-ready/model_reday_data/huc10_1706020806_24x24.zarr
Newly done: 91; total done now: 91 / 91


/Users/khaja/Documents/SnowML/.venv311/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
